# Findings

- MRO logic : How `TemporalBatchMixin` executes `forward`.
- `einops` : handling 5D tensors. 

In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 


TemporalBatchMixin.forward runs because ResNet5 has no forward, so the MRO finds the mixin's. ✓
Inside it, self._forward resolves via the MRO to ResNet5._forward (the subclass's version shadows the mixin's). ✓

Point 3 needs a small fix. The mixin's _forward stub is not about ordering — there's nothing that runs "before" the mixin, and it's not the mixin choosing not to execute something. The accurate picture is:
The mixin's _forward is a fallback that is only ever reached if a subclass forgot to define its own _forward. In normal operation — ResNet5 does define _forward — the mixin's _forward is completely shadowed and never executes at all. It's dead code in the happy path. It only becomes reachable when someone writes a subclass of the mixin but omits _forward; then self._forward falls through the MRO to the stub, which raises NotImplementedError to say "you were required to implement this and didn't."

In [ ]:
from einops import rearrange

class TemporalBatchMixin: 
    """
    Mixin class that handles automatic temporal batching for 4D/5D tensors. 

    This mixin provides a unified forward() method that: 
    - For 5D tensors [B, C, T, H, W]: flattens temporal dim, applies _forward(), restores shape
    - For 4D tensors [B, C, H, W]: directly applies _forward()

    Subclasses must implement _forward(self,x) for 4D tensors 
    """

    def _forward(self,x): 
        """
        Process 4D tensor [B, C, H, W]. MUst be implement by subclasses.

        Args:
            x: Input tensor of shape [B, C, H, W]

        Returns: 
            Output tensor of shape [B, C_out, H_out, W_out]
        """
        raise NotImplementedError("Subclasses must implement _forward()")
    
    def forward(self,x): 
        """
        Forward pass supporting both 4D and 5D tensors. 

        Args:
            x: Input tensor of shape [B, C, H, W] or [B, C, T, H, W]

        Returns: 
            Output tensor with same batch and temporal dimensions as input
        """
        assert x.ndim in [
            4,
            5
        ], "Supports only 4D [B, C, H, W] or 5D [B, C, T, H, W] tensors"
        if x.ndim == 5: 
            b = x.shape[0]
            x = rearrange(x, "b c t h w -> (b t) c h w")
            out = self._forward(x)
            out = rearrange(out, "(b t) c h w -> b c t h w", b = b)
            return out 
        else: 
            return self._forward(x)

In [21]:
from einops import rearrange

t = torch.randint(0,10,(2,1,10,64,64))

mutate_t = rearrange(t, "b c t h w -> (b t) c h w")
print(mutate_t.shape) 

unmutate_t = rearrange(mutate_t, "(b t) c h w -> b c t h w", b = 2)
print(unmutate_t.shape)

torch.Size([20, 1, 64, 64])
torch.Size([2, 1, 10, 64, 64])


In [ ]:
class ResNet5(TemporalBatchMixin,nn.Module): 
    pass 